In [1]:
import duckdb
con = duckdb.connect("olist.duckdb")
print("connected")
print(con.execute("SELECT COUNT(*) FROM delivery_orders").fetchone()[0], "rows in delivery_orders")

connected
96470 rows in delivery_orders


In [2]:
con.execute("""
CREATE OR REPLACE TABLE seller_revenue AS
WITH s AS (
  SELECT seller_id,
         COUNT(DISTINCT order_id) AS orders,
         ROUND(SUM(price), 2) AS revenue
  FROM item_level
  WHERE order_status = 'delivered'
  GROUP BY 1
),
r AS (
  SELECT *,
         ROW_NUMBER() OVER (ORDER BY revenue DESC, seller_id) AS rnk,
         COUNT(*) OVER () AS total_sellers,
         SUM(revenue) OVER () AS total_revenue,
         SUM(revenue) OVER (ORDER BY revenue DESC, seller_id
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_revenue
  FROM s
)
SELECT seller_id, orders, revenue, rnk, total_sellers,
       ROUND(total_revenue, 2) AS total_revenue,
       ROUND(100.0 * rnk / total_sellers, 2) AS pct_sellers,
       ROUND(100.0 * revenue / total_revenue, 3) AS revenue_share_pct,
       ROUND(100.0 * cum_revenue / total_revenue, 2) AS cum_revenue_pct
FROM r
ORDER BY rnk
""")

con.execute("SELECT * FROM seller_revenue LIMIT 10").df()

,seller_id,orders,revenue,rnk,total_sellers,total_revenue,pct_sellers,revenue_share_pct,cum_revenue_pct
0,4869f7a5dfa277a7dca6462dcf3b52b2,1124,226987.93,1,2970,13221498.11,0.03,1.717,1.72
1,53243585a1d6dc2643021fd1853d8905,348,217940.44,2,2970,13221498.11,0.07,1.648,3.37
2,4a3ca9315b744ce9f8e9374361493884,1772,196882.12,3,2970,13221498.11,0.10,1.489,4.85
3,fa1c13f2614d7b5c4749cbc52fecda94,578,190917.14,4,2970,13221498.11,0.13,1.444,6.30
4,7c67e1448b00f6e969d365cea6b010ab,973,186570.05,5,2970,13221498.11,0.17,1.411,7.71
5,7e93a43ef30c4f03f38b393420bc753a,319,165981.49,6,2970,13221498.11,0.20,1.255,8.96
6,da8622b14eb17ae2831f4ac5b9dab84a,1311,159816.87,7,2970,13221498.11,0.24,1.209,10.17
7,7a67c85e85bb2ce8582c35f2203ad736,1145,139658.69,8,2970,13221498.11,0.27,1.056,11.23
8,1025f0e2d44d7041d6cf58b6550e0bfa,910,138208.56,9,2970,13221498.11,0.30,1.045,12.28
9,955fee9216a65b617aa5c0531780ce60,1261,131836.71,10,2970,13221498.11,0.34,0.997,13.27


In [3]:
con.execute("""
CREATE OR REPLACE TABLE seller_concentration AS
SELECT MAX(total_sellers) AS sellers,
       MAX(total_revenue) AS total_revenue,
       MAX(CASE WHEN rnk <= 10 THEN cum_revenue_pct END) AS top10_sellers_revenue_pct,
       CAST(CEIL(0.10 * MAX(total_sellers)) AS INTEGER) AS top10pct_seller_count,
       MAX(CASE WHEN rnk <= CEIL(0.10 * total_sellers) THEN cum_revenue_pct END) AS top10pct_revenue_pct,
       MAX(CASE WHEN rnk <= CEIL(0.20 * total_sellers) THEN cum_revenue_pct END) AS top20pct_revenue_pct,
       MIN(CASE WHEN cum_revenue_pct >= 80 THEN rnk END) AS sellers_for_80pct,
       ROUND(100.0 * MIN(CASE WHEN cum_revenue_pct >= 80 THEN rnk END) / MAX(total_sellers), 2) AS sellers_for_80pct_share
FROM seller_revenue
""")

con.execute("SELECT * FROM seller_concentration").df()

,sellers,total_revenue,top10_sellers_revenue_pct,top10pct_seller_count,top10pct_revenue_pct,top20pct_revenue_pct,sellers_for_80pct,sellers_for_80pct_share
0,2970,13221498.11,13.27,297,67.11,82.29,533,17.95


In [4]:
con.execute("""
CREATE OR REPLACE TABLE seller_benchmark AS
SELECT COUNT(*) AS delivered_orders,
       ROUND(100.0 * SUM(CASE WHEN days_late > 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_pct,
       ROUND(100.0 * SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) / COUNT(review_score), 2) AS low_review_pct,
       ROUND(AVG(review_score), 2) AS avg_review
FROM delivery_orders
""")

con.execute("SELECT * FROM seller_benchmark").df()

,delivered_orders,late_pct,low_review_pct,avg_review
0,96470,6.77,12.77,4.16


In [5]:
con.execute("""
CREATE OR REPLACE TABLE seller_quality AS
WITH so AS (
  SELECT DISTINCT seller_id, order_id
  FROM item_level
  WHERE order_status = 'delivered'
),
j AS (
  SELECT so.seller_id, d.order_id, d.days_late, d.review_score
  FROM so
  JOIN delivery_orders d USING (order_id)
),
q AS (
  SELECT seller_id,
         COUNT(*) AS orders,
         SUM(CASE WHEN days_late > 0 THEN 1 ELSE 0 END) AS late_orders,
         ROUND(100.0 * SUM(CASE WHEN days_late > 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_pct,
         COUNT(review_score) AS reviewed_orders,
         ROUND(AVG(review_score), 2) AS avg_review,
         ROUND(100.0 * SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) / COUNT(review_score), 2) AS low_review_pct
  FROM j
  GROUP BY 1
  HAVING COUNT(*) >= 30
)
SELECT q.*,
       r.revenue,
       r.revenue_share_pct,
       COALESCE(q.late_pct > b.late_pct AND q.low_review_pct >= 2 * b.low_review_pct, FALSE) AS flagged
FROM q
LEFT JOIN seller_revenue r USING (seller_id)
CROSS JOIN seller_benchmark b
ORDER BY q.low_review_pct DESC
""")

con.execute("SELECT * FROM seller_quality LIMIT 10").df()

,seller_id,orders,late_orders,late_pct,reviewed_orders,avg_review,low_review_pct,revenue,revenue_share_pct,flagged
0,1ca7077d890b907f89be8c954a02686a,108,18.0,16.67,107,2.39,59.81,12474.64,0.094,True
1,2eb70248d66e0e3ef83659f71b244378,187,21.0,11.23,184,2.79,47.28,38990.72,0.295,True
2,972d0f9cf61b499a4812cf0bfa3ad3c4,81,9.0,11.11,79,3.03,41.77,7910.42,0.060,True
3,a49928bcdf77c55c6d6e05e09a9b4ca5,96,21.0,21.88,96,3.03,40.63,8646.90,0.065,True
4,54965bbe3e4f07ae045b90b0b8541f52,73,22.0,30.14,69,3.14,40.58,10351.70,0.078,True
5,bbad7e518d7af88a0897397ffdca1979,68,11.0,16.18,67,3.18,40.30,4426.32,0.033,True
6,6fd52c528dcb38be2eea044946b811f8,67,9.0,13.43,66,3.39,36.36,7743.02,0.059,True
7,2a1348e9addc1af5aaa619b1a3679d6b,48,13.0,27.08,47,3.17,36.17,2697.40,0.020,True
8,712e6ed8aa4aa1fa65dab41fed5737e4,77,16.0,20.78,76,3.39,35.53,39385.00,0.298,True
9,0b35c634521043bf4b47e21547b99ab5,50,8.0,16.00,49,3.41,34.69,13235.65,0.100,True


In [6]:
con.execute("""
CREATE OR REPLACE TABLE seller_impact AS
WITH f AS (
  SELECT seller_id FROM seller_quality WHERE flagged
),
fo AS (
  SELECT DISTINCT i.order_id
  FROM item_level i
  JOIN f USING (seller_id)
  WHERE i.order_status = 'delivered'
)
SELECT (SELECT COUNT(*) FROM f) AS flagged_sellers,
       (SELECT COUNT(*) FROM seller_quality) AS sellers_30plus,
       COALESCE((SELECT ROUND(SUM(revenue_share_pct), 2) FROM seller_quality WHERE flagged), 0) AS flagged_revenue_share_pct,
       (SELECT COUNT(*) FROM delivery_orders d JOIN fo USING (order_id) WHERE d.days_late > 0) AS flagged_late_orders,
       (SELECT COUNT(*) FROM delivery_orders WHERE days_late > 0) AS all_late_orders,
       (SELECT COUNT(*) FROM delivery_orders d JOIN fo USING (order_id) WHERE d.review_score <= 2) AS flagged_low_orders,
       (SELECT COUNT(*) FROM delivery_orders WHERE review_score <= 2) AS all_low_orders
""")

con.execute("SELECT * FROM seller_impact").df()

,flagged_sellers,sellers_30plus,flagged_revenue_share_pct,flagged_late_orders,all_late_orders,flagged_low_orders,all_low_orders
0,25,627,2.03,321,6534,687,12236


In [7]:
con.execute("""
SELECT
  (SELECT ROUND(SUM(price), 2) FROM item_level WHERE order_status = 'delivered') AS delivered_revenue,
  (SELECT ROUND(SUM(revenue), 2) FROM seller_revenue) AS seller_revenue_total,
  (SELECT COUNT(*) FROM seller_revenue) AS sellers,
  (SELECT COUNT(DISTINCT seller_id) FROM seller_revenue) AS distinct_sellers,
  (SELECT MAX(cum_revenue_pct) FROM seller_revenue) AS final_cum_pct,
  (SELECT COUNT(*) FROM seller_quality) AS sellers_30plus
""").df()

,delivered_revenue,seller_revenue_total,sellers,distinct_sellers,final_cum_pct,sellers_30plus
0,13221498.11,13221498.11,2970,2970,100.0,627


In [8]:
con.execute("COPY seller_revenue TO 'dashboard/seller_revenue.csv' (HEADER, DELIMITER ',')")
con.execute("COPY seller_concentration TO 'dashboard/seller_concentration.csv' (HEADER, DELIMITER ',')")
con.execute("COPY seller_quality TO 'dashboard/seller_quality.csv' (HEADER, DELIMITER ',')")
con.execute("COPY seller_benchmark TO 'dashboard/seller_benchmark.csv' (HEADER, DELIMITER ',')")
con.execute("COPY seller_impact TO 'dashboard/seller_impact.csv' (HEADER, DELIMITER ',')")
print("saved")

saved


In [9]:
c = con.execute("SELECT * FROM seller_concentration").df().iloc[0]
b = con.execute("SELECT * FROM seller_benchmark").df().iloc[0]
imp = con.execute("SELECT * FROM seller_impact").df().iloc[0]
q = con.execute("SELECT * FROM seller_quality").df()

In [10]:
share_q = round(q.revenue_share_pct.sum(), 1)

In [12]:
if c.top10pct_revenue_pct >= 60:
    verdict7 = "Revenue is highly concentrated in a small group of sellers, so losing a few of them would noticeably hit revenue."
else:
    verdict7 = "Revenue is moderately concentrated, with a long tail of smaller sellers."

print("=== COPY THIS INTO business_questions.md (Q7 answer) ===")
print(f"- Answer: {int(c.sellers):,} sellers sold delivered items. The top 10 sellers account for {c.top10_sellers_revenue_pct}% of revenue, the top 10% of sellers ({int(c.top10pct_seller_count)} sellers) for {c.top10pct_revenue_pct}% and the top 20% for {c.top20pct_revenue_pct}%. "
      f"It takes {int(c.sellers_for_80pct)} sellers ({c.sellers_for_80pct_share}% of all sellers) to reach 80% of revenue. {verdict7}")
print()

worst3 = q.nlargest(3, "low_review_pct")
worst_txt = ", ".join(f"seller ID starting {r.seller_id[:8]} ({int(r.orders)} orders, {r.low_review_pct}% low reviews, {r.late_pct}% late)" for r in worst3.itertuples())

if imp.flagged_sellers > 0:
    flag_txt = (f"{int(imp.flagged_sellers)} sellers are flagged because their late rate is above the marketplace rate and their low-review rate is at least double it. "
                f"They produce {imp.flagged_revenue_share_pct}% of revenue, but their orders account for {round(100 * imp.flagged_late_orders / imp.all_late_orders, 1)}% of all late orders and {round(100 * imp.flagged_low_orders / imp.all_low_orders, 1)}% of all 1-2 star orders.")
else:
    flag_txt = "No seller meets the flag rule (late rate above the marketplace rate and low-review rate at least double it)."

print("=== COPY THIS INTO business_questions.md (Q8 answer) ===")
print(f"- Answer: {len(q)} sellers have at least 30 delivered orders and together produce {share_q}% of revenue. The marketplace late rate is {b.late_pct}% and the low-review rate is {b.low_review_pct}%. "
      f"The worst sellers by low-review rate are {worst_txt}. {flag_txt} "
      f"These sellers are candidates for warnings, coaching or removal; rates for sellers with few orders are noisy.")
print()
print("=== COPY THIS INTO README.md (under ## Method) ===")
print("- Seller concentration uses delivered-order item revenue per seller. Seller quality uses sellers with at least 30 delivered orders; an order with items from several sellers counts toward each of them. A seller is flagged when its late rate is above the marketplace rate and its low-review rate (1-2 stars) is at least double the marketplace rate. Olist provides seller IDs but no seller names.")

=== COPY THIS INTO business_questions.md (Q7 answer) ===
- Answer: 2,970 sellers sold delivered items. The top 10 sellers account for 13.27% of revenue, the top 10% of sellers (297 sellers) for 67.11% and the top 20% for 82.29%. It takes 533 sellers (17.95% of all sellers) to reach 80% of revenue. Revenue is highly concentrated in a small group of sellers, so losing a few of them would noticeably hit revenue.

=== COPY THIS INTO business_questions.md (Q8 answer) ===
- Answer: 627 sellers have at least 30 delivered orders and together produce 77.1% of revenue. The marketplace late rate is 6.77% and the low-review rate is 12.77%. The worst sellers by low-review rate are seller ID starting 1ca7077d (108 orders, 59.81% low reviews, 16.67% late), seller ID starting 2eb70248 (187 orders, 47.28% low reviews, 11.23% late), seller ID starting 972d0f9c (81 orders, 41.77% low reviews, 11.11% late). 25 sellers are flagged because their late rate is above the marketplace rate and their low-review r